# Task 5 - Source metadata ingestion into MongoDB

Spark reads only `cpg.source-metadata.v1`, parses an explicit
schema, and writes replacement upserts keyed by `_id=file_id`. Its checkpoint
is retained on a Docker volume.

## Why metadata goes through Spark

Metadata is naturally modeled as one current document per source file, unlike
the many nodes and edges stored in Neo4j. The streaming query therefore reads
only `cpg.source-metadata.v1`, applies an explicit nested schema, and uses the
MongoDB connector to replace/upsert `_id=file_id`. Replacement is important:
an append-only collection would preserve history but would fail the lab's
no-duplication final-state requirement, while partial updates could leave stale
fields behind.

Spark owns progress through a persistent checkpoint volume instead of through a
custom offset table. `startingOffsets=earliest` matters only when that checkpoint
is empty; after the first run, the saved offsets decide where processing resumes.
Each MongoDB document also records its Kafka offset, which makes the relationship
between source progress and the visible document inspectable.

A custom `foreachBatch` writer could implement the same policy, but the official
MongoDB Spark Connector already provides replacement upserts and reduces the
amount of recovery code we would have to maintain. The explicit schema and
`read_committed` isolation trade some flexibility for earlier detection of
incompatible or uncommitted events.

In [1]:
import json
import sys
from pathlib import Path

root = Path('..').resolve()
sys.path.insert(0, str(root / 'scripts'))
from capture_replay_evidence import mongo_snapshot

snapshot = mongo_snapshot('867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7')
print(json.dumps(snapshot, indent=2))
assert snapshot['documents'] == snapshot['distinct_files'] == 61
assert snapshot['document']['_id'] == '867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7'
print('PASS: MongoDB has one replacement-upserted document per file')

{
  "documents": 61,
  "distinct_files": 61,
  "document": {
    "_id": "867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7",
    "path": "optimum/version.py",
    "content_hash": "079eca803ea5bfe068bc805997b013cc14c9ddc8629a2ec634d12bcebb6720ce",
    "node_counts": {
      "AST": 25,
      "SYNTHETIC": 4
    },
    "edge_counts": {
      "AST": 24,
      "CFG": 9,
      "DFG": 3
    },
    "processed_at": "2026-07-24T15:09:20.140539Z",
    "run_id": "de2b478f74c345138e77a636e2d2d660",
    "kafka_offset": 416
  },
  "other_documents_count": 60,
  "other_documents_digest": "80b1e9f81a9a9d42c67f1195204b990cac32ae1ed60665c5a1e68814a7be5cc1"
}
PASS: MongoDB has one replacement-upserted document per file


In [2]:
import json
import sys
from pathlib import Path

root = Path('..').resolve()
sys.path.insert(0, str(root / 'scripts'))
from capture_replay_evidence import checkpoint_offset, kafka_end_offset

progress = {
    'spark_checkpoint_offset': checkpoint_offset(),
    'metadata_topic_end_offset': kafka_end_offset('cpg.source-metadata.v1'),
}
print(json.dumps(progress, indent=2))
assert progress['spark_checkpoint_offset'] == progress['metadata_topic_end_offset']
print('PASS: Spark checkpoint has consumed the metadata topic')

{
  "spark_checkpoint_offset": 418,
  "metadata_topic_end_offset": 418
}
PASS: Spark checkpoint has consumed the metadata topic


![MongoDB UI capture of the replay file; the executable output above verifies the final live offset and checkpoint](figures/mongodb-ui.png)

*MongoDB UI capture of the replay file; the executable output above verifies the final live offset and checkpoint*

## Reflection

The final collection contains one `_id=file_id` document for
each processed source file, and the checkpoint reaches the metadata topic's end
offset. First startup was noticeably slower because Spark had to resolve the
Kafka and MongoDB packages; persisting the Ivy cache removed that repeated cost.
We also learned that a document count alone is weak evidence: the same count can
hide accidental rewrites or stale content. The chapter now pairs distinct-ID
counts with the replay file's content hash, Kafka offset, and checkpoint so the
replacement behavior can be verified rather than inferred.